# Validation plots — single-stage transformer with predicted-origin flip

Based on `validate_transformer.py`.  Three scenarios are compared:

- **Nominal** — original track features, no flip.
- **Hard-flip** — signed IP features negated for tracks whose *true* `GN2v01_trackOrigin` ∈ `FLIP_ORIGINS` (same as training).
- **Pred-flip** — signed IP features negated for tracks where the model's own predicted origin probability satisfies `p_from_b + p_from_bc > FLIP_THRESHOLD`.

Workflow:
1. Run the *config*, *definitions*, and *inference* cells once.
   - If a results cache exists in `PLOT_DIR` it is loaded; set `FORCE_RECOMPUTE = True` to override.
2. Tune and re-run any plotting cell as many times as you like.

In [ ]:
import glob
import hashlib
import json
import os
import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt

## Config

Edit these values and re-run.

In [ ]:
_DEFAULTS = {
    "data_file":      "mc-flavtag-ttbar-small.h5",
    "model_file":     "/Users/adminbingxuanliu/Work/SALTOpenData/FlipStudies/results/transformer_results_nominal/transformer_jet_classifier_nominal.pt",
    "val_plot_dir":   "./transformer_results_val_pred_flip/",
    "val_cache_dir":  "/Users/adminbingxuanliu/Work/SALTOpenData/FlipStudies/results/val_cache",
    "n_skip":         12_000_000,
    "n_val":          1_200_000,
    "top_k":          40,
    "batch_size":     1024,
    "n_origins":      8,
    "d_model":        32,
    "n_heads":        2,
    "n_layers":       2,
    "d_ffn":          64,
    "dropout":        0.1,
    # Threshold on p_from_b + p_from_bc for the predicted-origin flip
    "flip_threshold": 0.9,
    "track_fields": [
        "qOverP", "deta", "dphi", "d0", "z0SinTheta",
        "qOverPUncertainty", "thetaUncertainty", "phiUncertainty",
        "lifetimeSignedD0Significance", "lifetimeSignedZ0SinThetaSignificance",
        "numberOfPixelHits", "numberOfSCTHits",
        "numberOfInnermostPixelLayerHits", "numberOfNextToInnermostPixelLayerHits",
        "numberOfInnermostPixelLayerSharedHits", "numberOfInnermostPixelLayerSplitHits",
        "numberOfPixelSharedHits", "numberOfPixelSplitHits", "numberOfSCTSharedHits",
    ],
    "flip_fields": [
        "lifetimeSignedD0Significance", "lifetimeSignedZ0SinThetaSignificance",
        "d0", "z0SinTheta",
    ],
    # Hard-flip reference: true origins to flip
    # 0=Pileup 1=Fake 2=Primary 3=From b 4=From b->c 5=From c 6=From tau 7=Other secondary
    "flip_origins":     [3, 4],
    "flavour_to_label": {"5": 0, "4": 1, "0": 2},
    "class_names":      ["b-jet", "c-jet", "light-jet"],
    "colours":          {"b-jet": "#1f77b4", "c-jet": "#ff7f0e", "light-jet": "#2ca02c"},
}

CONFIG_FILE = None

cfg = dict(_DEFAULTS)
if CONFIG_FILE is not None:
    with open(CONFIG_FILE) as _f:
        cfg.update({k: v for k, v in json.load(_f).items() if k in _DEFAULTS})

DATA_FILE        = cfg["data_file"]
MODEL_FILE       = cfg["model_file"]
PLOT_DIR         = cfg["val_plot_dir"]
CACHE_DIR        = cfg["val_cache_dir"]
N_SKIP           = cfg["n_skip"]
N_VAL            = cfg["n_val"]
TOP_K            = cfg["top_k"]
BATCH_SIZE       = cfg["batch_size"]
N_ORIGINS        = cfg["n_origins"]
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
FLIP_THRESHOLD   = cfg["flip_threshold"]
D_MODEL          = cfg["d_model"]
N_HEADS          = cfg["n_heads"]
N_LAYERS         = cfg["n_layers"]
D_FFN            = cfg["d_ffn"]
DROPOUT          = cfg["dropout"]
TRACK_FIELDS     = cfg["track_fields"]
FLIP_FIELDS      = cfg["flip_fields"]
FLIP_ORIGINS     = cfg["flip_origins"]
FLAVOUR_TO_LABEL = {int(k): v for k, v in cfg["flavour_to_label"].items()}
CLASS_NAMES      = cfg["class_names"]
COLOURS          = cfg["colours"]

N_FEATS = len(TRACK_FIELDS)
FLIP_FEAT_IDX        = [TRACK_FIELDS.index(f) for f in FLIP_FIELDS]
PRED_FLIP_ORIGIN_IDX = [3, 4]  # From b, From b->c

os.makedirs(PLOT_DIR,  exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

FORCE_RECOMPUTE = False

# Threshold-independent cache: nominal + hard-flip model outputs, p_b_sum, X_val, mask_val.
# Rebuilt only when FORCE_RECOMPUTE=True or missing.
RESULTS_CACHE   = os.path.join(PLOT_DIR, "val_results_cache.npz")

# Threshold-specific cache: just all_probs_pred_flip.
# A new threshold automatically uses a different file — no stale results.
PRED_FLIP_CACHE = os.path.join(PLOT_DIR,
                                f"val_pred_flip_probs_thr{int(FLIP_THRESHOLD * 100):03d}.npz")

VAL_IDX_CACHE   = os.path.join(CACHE_DIR, "val_idx.npy")

print(f"Predicted-flip threshold : p_from_b + p_from_bc > {FLIP_THRESHOLD}")
print(f"Hard-flip origins        : {FLIP_ORIGINS}")
print(f"Base results cache       : {RESULTS_CACHE}")
print(f"Pred-flip cache          : {PRED_FLIP_CACHE}")

## Data loading and model definitions

In [ ]:
def _cache_key(idx, flip):
    origins_tag = "all" if FLIP_ORIGINS is None else "".join(str(o) for o in sorted(FLIP_ORIGINS))
    h   = hashlib.md5(idx.tobytes()).hexdigest()[:12]
    tag = f"flip{origins_tag}" if flip else "nom"
    return os.path.join(CACHE_DIR, f"tracks_{h}_{tag}.npz")


def load_tracks(path, idx, flip=False):
    """Returns (N, K, F) features, (N, K) validity mask, (N,) labels, (N, K) origins.
    When flip=True, signed IP features are negated for tracks in FLIP_ORIGINS.
    Both nominal and flipped data are cached."""
    cp = _cache_key(idx, flip)
    if os.path.exists(cp):
        d = np.load(cp)
        if "origins" in d:
            return d["X"], d["mask"], d["y"], d["origins"]

    with h5py.File(path, "r") as f:
        flavour_id = f["jets"]["HadronConeExclTruthLabelID"][idx]
        keep_jet   = np.isin(flavour_id, list(FLAVOUR_TO_LABEL.keys()))
        fidx       = idx[keep_jet]

        valid  = f["tracks"]["valid"][fidx]
        d0     = f["tracks"]["d0"][fidx].astype(np.float32)
        ip2d   = f["tracks"]["lifetimeSignedD0Significance"][fidx].astype(np.float32)
        origin = f["tracks"]["GN2v01_trackOrigin"][fidx].astype(np.int8)
        arrs   = {fld: f["tracks"][fld][fidx].astype(np.float32) for fld in TRACK_FIELDS}

    keep = valid & (np.abs(d0) < 3.5)

    if flip:
        flip_mask = (np.ones_like(origin, dtype=bool) if FLIP_ORIGINS is None
                     else np.isin(origin, FLIP_ORIGINS))
        ip2d_sort = ip2d.copy()
        ip2d_sort[flip_mask] = -ip2d_sort[flip_mask]
    else:
        flip_mask = np.zeros_like(origin, dtype=bool)
        ip2d_sort = ip2d

    sort_key = ip2d_sort.copy()
    sort_key[~keep] = -np.inf
    order = np.argsort(-sort_key, axis=1)

    feat_list = []
    for fld in TRACK_FIELDS:
        arr = arrs[fld].copy()
        if flip and fld in FLIP_FIELDS:
            arr[flip_mask] = -arr[flip_mask]
        feat_list.append(arr)
    feats = np.stack(feat_list, axis=-1)

    topk_idx    = order[:, :TOP_K]
    rows        = np.arange(len(fidx))[:, None]
    topk_feat   = feats[rows, topk_idx]
    topk_valid  = keep[rows, topk_idx]
    topk_feat   = np.where(topk_valid[:, :, None], topk_feat, 0.0).astype(np.float32)
    topk_origin = origin[rows, topk_idx].astype(np.int64)
    topk_origin[~topk_valid] = -1

    labels = np.array([FLAVOUR_TO_LABEL[v] for v in flavour_id[keep_jet]], dtype=np.int64)

    np.savez(cp, X=topk_feat, mask=topk_valid, y=labels, origins=topk_origin)
    return topk_feat, topk_valid, labels, topk_origin


class JetDataset(Dataset):
    def __init__(self, X, mask, y, origins):
        self.X       = torch.from_numpy(X)
        self.mask    = torch.from_numpy(mask)
        self.y       = torch.from_numpy(y)
        self.origins = torch.from_numpy(origins)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.mask[i], self.y[i], self.origins[i]

In [ ]:
class JetTransformer(nn.Module):
    def __init__(self, in_dim, d_model, n_heads, n_layers, d_ffn, dropout,
                 n_classes, n_origins):
        super().__init__()
        self.input_proj  = nn.Linear(in_dim, d_model)
        self.cls_token   = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ffn,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder     = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.classifier  = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, n_classes))
        self.origin_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, n_origins))

    def forward(self, x, mask):
        B   = x.size(0)
        h   = self.input_proj(x)
        cls = self.cls_token.expand(B, -1, -1)
        h   = torch.cat([cls, h], dim=1)
        cls_valid            = torch.ones(B, 1, dtype=torch.bool, device=x.device)
        src_key_padding_mask = ~torch.cat([cls_valid, mask], dim=1)
        h = self.encoder(h, src_key_padding_mask=src_key_padding_mask)
        return self.classifier(h[:, 0]), self.origin_head(h[:, 1:])


def run_inference(model, loader, collect_origins=False):
    """Run jet-level inference.
    When collect_origins=True, also returns per-track origin argmax predictions,
    true origins (valid tracks only), and p_b_sum (N, K) = p_from_b + p_from_bc."""
    preds, trues, probs = [], [], []
    origin_preds_list, origin_trues_list, p_b_sums = [], [], []
    with torch.no_grad():
        for X_b, mask_b, y_b, orig_b in loader:
            X_b, mask_b, y_b = X_b.to(DEVICE), mask_b.to(DEVICE), y_b.to(DEVICE)
            logits, track_logits = model(X_b, mask_b)
            preds.append(logits.argmax(dim=1).cpu())
            trues.append(y_b.cpu())
            probs.append(torch.softmax(logits, dim=1).cpu())
            if collect_origins:
                origin_softmax = torch.softmax(track_logits, dim=-1)   # (B, K, 8)
                p_b_sum = sum(origin_softmax[..., i] for i in PRED_FLIP_ORIGIN_IDX)  # (B, K)
                p_b_sums.append(p_b_sum.cpu())
                origin_preds_list.append(track_logits.argmax(dim=-1).cpu())
                origin_trues_list.append(orig_b)
    result = (torch.cat(preds).numpy(),
              torch.cat(trues).numpy(),
              torch.cat(probs).numpy())
    if collect_origins:
        op = torch.cat(origin_preds_list).numpy().ravel()
        ot = torch.cat(origin_trues_list).numpy().ravel()
        valid_tracks = ot >= 0
        pb = torch.cat(p_b_sums).numpy()   # (N, K) — all tracks including padding
        return result + (op[valid_tracks], ot[valid_tracks], pb)
    return result


def _apply_flip(X_nom, mask_nom, flip_track_mask):
    """Negate FLIP_FIELDS in-place for tracks selected by flip_track_mask (N, K).
    Track order is unchanged from the nominal sort (no re-sorting)."""
    X_flip = X_nom.copy()
    for fi in FLIP_FEAT_IDX:
        X_flip[:, :, fi] = np.where(flip_track_mask, -X_flip[:, :, fi], X_flip[:, :, fi])
    return X_flip


def build_hard_flip_features(X_nom, mask_nom, origins):
    """Negate FLIP_FIELDS for valid tracks whose true origin is in FLIP_ORIGINS."""
    if FLIP_ORIGINS is None:
        flip_track_mask = mask_nom
    else:
        flip_track_mask = np.isin(origins, FLIP_ORIGINS) & mask_nom
    n_flipped = flip_track_mask.sum()
    n_valid   = mask_nom.sum()
    print(f"Hard flip : {n_flipped:,} / {n_valid:,} valid tracks flipped "
          f"({100*n_flipped/n_valid:.1f}%)")
    return _apply_flip(X_nom, mask_nom, flip_track_mask)


def build_pred_flip_features(X_nom, mask_nom, p_b_sum, flip_threshold):
    """Negate FLIP_FIELDS for valid tracks where p_from_b + p_from_bc > flip_threshold."""
    flip_track_mask = (p_b_sum > flip_threshold) & mask_nom
    n_flipped = flip_track_mask.sum()
    n_valid   = mask_nom.sum()
    print(f"Pred flip : {n_flipped:,} / {n_valid:,} valid tracks flipped "
          f"({100*n_flipped/n_valid:.1f}%) at threshold {flip_threshold}")
    return _apply_flip(X_nom, mask_nom, flip_track_mask)

## Run inference (two-stage caching)

Two separate caches are used:

**`RESULTS_CACHE`** — threshold-independent. Stores nominal + hard-flip model outputs,
`p_b_sum` (per-track b-probability from the nominal pass), `X_val`, and `mask_val`.
Built once and reused regardless of `FLIP_THRESHOLD`.

**`PRED_FLIP_CACHE`** — threshold-specific (filename encodes the threshold).
Stores only `all_probs_pred_flip`. A new threshold automatically uses a fresh file —
no stale results, no need to set `FORCE_RECOMPUTE`.

Both flip variants (hard and predicted) are built **on the fly** from the cached nominal
features — no separate flip track files are read or written.

In [ ]:
# ── Stage 1: threshold-independent base results ────────────────────────
if os.path.exists(RESULTS_CACHE) and not FORCE_RECOMPUTE:
    print(f"Loading base cache from {RESULTS_CACHE}")
    _d = np.load(RESULTS_CACHE)
    all_preds    = _d["all_preds"]
    all_true     = _d["all_true"]
    all_probs    = _d["all_probs"]
    all_probs_flip = _d["all_probs_flip"]
    origin_preds = _d["origin_preds"]
    origin_true  = _d["origin_true"]
    p_b_sum      = _d["p_b_sum"]
    X_val        = _d["X_val"]
    mask_val     = _d["mask_val"]
else:
    # Load nominal tracks only — check for track cache first
    nom_files = glob.glob(os.path.join(CACHE_DIR, "tracks_*_nom.npz"))
    if len(nom_files) == 1 and not FORCE_RECOMPUTE:
        print(f"Loading nominal track cache: {nom_files[0]}")
        _nom = np.load(nom_files[0])
        X_val, mask_val, y_val, origins_val = _nom["X"], _nom["mask"], _nom["y"], _nom["origins"]
    else:
        if os.path.exists(VAL_IDX_CACHE) and not FORCE_RECOMPUTE:
            print(f"Loading val_idx from {VAL_IDX_CACHE}")
            val_idx = np.load(VAL_IDX_CACHE)
        else:
            print("Computing val_idx from data file...")
            rng = np.random.default_rng(42)
            with h5py.File(DATA_FILE, "r") as f:
                n_total = f["jets"].shape[0]
            all_idx = rng.permutation(n_total)
            val_idx = np.sort(all_idx[N_SKIP:N_SKIP + N_VAL])
            np.save(VAL_IDX_CACHE, val_idx)
            print(f"Saved val_idx to {VAL_IDX_CACHE}")
        print("Loading nominal validation data...")
        X_val, mask_val, y_val, origins_val = load_tracks(DATA_FILE, val_idx, flip=False)

    print(f"Validation jets: {len(y_val):,}  "
          f"(b={(y_val==0).sum():,}  c={(y_val==1).sum():,}  light={(y_val==2).sum():,})")

    # Build hard-flip and pred-flip features on the fly from nominal tracks
    print("Building hard-flip features on the fly (no re-sort)...")
    X_val_flip = build_hard_flip_features(X_val, mask_val, origins_val)

    # Load model
    model = JetTransformer(N_FEATS, D_MODEL, N_HEADS, N_LAYERS, D_FFN, DROPOUT,
                           n_classes=3, n_origins=N_ORIGINS).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_FILE, map_location=DEVICE))
    model.eval()
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Loaded {MODEL_FILE}  ({n_params:,} parameters)")

    # Pass 1: nominal — also collects per-track origin probs
    print("Running inference on nominal inputs...")
    nom_loader = DataLoader(JetDataset(X_val, mask_val, y_val, origins_val),
                            batch_size=BATCH_SIZE)
    all_preds, all_true, all_probs, origin_preds, origin_true, p_b_sum = \
        run_inference(model, nom_loader, collect_origins=True)

    # Pass 2: hard-flip
    print("Running inference on hard-flip inputs...")
    flip_loader = DataLoader(JetDataset(X_val_flip, mask_val, y_val, origins_val),
                             batch_size=BATCH_SIZE)
    _, _, all_probs_flip = run_inference(model, flip_loader)

    np.savez(RESULTS_CACHE,
             all_preds=all_preds, all_true=all_true,
             all_probs=all_probs, all_probs_flip=all_probs_flip,
             origin_preds=origin_preds, origin_true=origin_true,
             p_b_sum=p_b_sum, X_val=X_val, mask_val=mask_val)
    print(f"Saved base cache to {RESULTS_CACHE}")

# ── Stage 2: threshold-specific pred-flip results ──────────────────────
# all_probs_pred_flip is always derived from the cached X_val + p_b_sum,
# never from a flip track cache. The cache file name encodes the threshold
# so changing FLIP_THRESHOLD automatically picks up a fresh file.
if os.path.exists(PRED_FLIP_CACHE) and not FORCE_RECOMPUTE:
    print(f"Loading pred-flip cache from {PRED_FLIP_CACHE}")
    all_probs_pred_flip = np.load(PRED_FLIP_CACHE)["all_probs_pred_flip"]
else:
    print(f"Building pred-flip features (threshold={FLIP_THRESHOLD})...")
    X_val_pred_flip = build_pred_flip_features(X_val, mask_val, p_b_sum, FLIP_THRESHOLD)
    # Load model if not already in memory (e.g. when base cache was loaded above)
    if "model" not in dir():
        model = JetTransformer(N_FEATS, D_MODEL, N_HEADS, N_LAYERS, D_FFN, DROPOUT,
                               n_classes=3, n_origins=N_ORIGINS).to(DEVICE)
        model.load_state_dict(torch.load(MODEL_FILE, map_location=DEVICE))
        model.eval()
        print(f"Loaded {MODEL_FILE}")
    print("Running inference on pred-flip inputs...")
    # y and origins are needed for DataLoader; origins unused (no collect_origins)
    dummy_origins = np.zeros((len(all_true), TOP_K), dtype=np.int64)
    pred_flip_loader = DataLoader(
        JetDataset(X_val_pred_flip, mask_val, all_true, dummy_origins),
        batch_size=BATCH_SIZE)
    _, _, all_probs_pred_flip = run_inference(model, pred_flip_loader)
    np.savez(PRED_FLIP_CACHE, all_probs_pred_flip=all_probs_pred_flip)
    print(f"Saved pred-flip cache to {PRED_FLIP_CACHE}")

acc = (all_preds == all_true).mean()
print(f"\nAccuracy (nominal): {acc:.4f}")
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES))
print("Confusion matrix:")
print(confusion_matrix(all_true, all_preds))

## Derived quantities (discriminant, cases)

Re-run this cell if you change the discriminant definition or FLIP_THRESHOLD.

In [ ]:
def make_disc(probs):
    pb, pc, pu = probs[:, 0], probs[:, 1], probs[:, 2]
    return np.log(pb / (0.2 * pc + 0.8 * pu + 1e-10))

disc           = make_disc(all_probs)           # Nominal
disc_flip      = make_disc(all_probs_flip)      # Hard-flip (true origins)
disc_pred_flip = make_disc(all_probs_pred_flip) # Predicted-origin flip

CASES = [
    ("Nominal",    all_probs,           disc),
    ("Hard-flip",  all_probs_flip,      disc_flip),
    ("Pred-flip",  all_probs_pred_flip, disc_pred_flip),
]
LINESTYLES  = {"Nominal": "-", "Hard-flip": "--", "Pred-flip": ":"}
CASE_COLOURS = {"Nominal": "#1f77b4", "Hard-flip": "#d62728", "Pred-flip": "#2ca02c"}

## Plot: per-track predicted b-probability distribution

Shows the distribution of `p_from_b + p_from_bc` for valid tracks in each jet flavour.
The vertical line marks `FLIP_THRESHOLD`.

In [ ]:
# p_b_sum shape: (N_jets, TOP_K)
labels_rep  = np.repeat(all_true, TOP_K)      # (N*K,)
nonzero     = mask_val.ravel()                 # valid track mask (N*K,)
p_b_flat    = p_b_sum.ravel()                  # (N*K,)

LINESTYLES_F = ['solid', 'dashed', 'dotted']
fig, ax = plt.subplots(figsize=(7, 5))
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    m = (labels_rep == cls_idx) & nonzero
    ax.hist(p_b_flat[m], bins=60, range=(0, 1), histtype="step",
            label=cls_name, color=COLOURS[cls_name], linewidth=3,
            density=True, linestyle=LINESTYLES_F[cls_idx])
ax.axvline(FLIP_THRESHOLD, color="black", linewidth=1.2, linestyle="--")
ax.set_xlabel(r"$p_{b} + p_{b{\to}c}$",
              loc="right", fontsize=12)
ax.set_ylabel("Density", loc="top", fontsize=12)
ax.set_yscale("log")
ax.legend(fontsize=12, loc = "lower left", bbox_to_anchor=(0.1, 0.05))
plt.tight_layout()
plt.savefig(PLOT_DIR + "val_p_b_sum_distribution.png", dpi=150, bbox_inches="tight")
print("Saved val_p_b_sum_distribution.png")

## Plot: fraction of tracks flipped vs threshold

Scans a range of thresholds and shows what fraction of valid tracks in each flavour would be flipped.

In [ ]:
thresholds = np.linspace(0, 1, 101)
fig, ax = plt.subplots(figsize=(7, 5))
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    m = (labels_rep == cls_idx) & nonzero
    vals = p_b_flat[m]
    frac = [(vals > thr).mean() for thr in thresholds]
    ax.plot(thresholds, frac, color=COLOURS[cls_name], linewidth=3,
            linestyle=LINESTYLES_F[cls_idx], label=cls_name)
ax.axvline(FLIP_THRESHOLD, color="black", linewidth=1.2, linestyle="--")
ax.set_xlabel(r"$p_{b} + p_{b{\to}c}$ threshold",
              loc="right", fontsize=12)
ax.set_ylabel("Fraction of flipped tracks", loc="top", fontsize=12)
ax.legend(fontsize=12)
ax.set_yscale("log")
ax.set_xscale("log")
ax.set_ylim(0.001, 1)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(PLOT_DIR + "val_flip_fraction_vs_threshold.png", dpi=150, bbox_inches="tight")
print("Saved val_flip_fraction_vs_threshold.png")

## Plot: track origin confusion matrix (nominal inputs)

In [ ]:
ORIGIN_NAMES = ["Pileup", "Fake", "Primary", "From b", "From b→c", "From c", "From τ", "Other sec."]
present      = sorted(np.unique(origin_true))
labels_pres  = [ORIGIN_NAMES[i] for i in present]

cm_orig = confusion_matrix(origin_true, origin_preds, labels=present, normalize="true")
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm_orig, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(present))); ax.set_yticks(range(len(present)))
ax.set_xticklabels(labels_pres, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(labels_pres, fontsize=8)
ax.set_xlabel("Predicted origin",fontsize=12, loc = "right"); ax.set_ylabel("True origin", fontsize=12, loc = "top")
for i in range(len(present)):
    for j in range(len(present)):
        ax.text(j, i, f"{cm_orig[i,j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if cm_orig[i,j] > 0.5 else "black")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(PLOT_DIR + "val_origin_confusion.png", dpi=150, bbox_inches="tight")
print("Saved val_origin_confusion.png")

## Plot: jet confusion matrix (nominal)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(all_true, all_preds, normalize="true")
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted", loc = "right", fontsize = 12); ax.set_ylabel("True", loc = "top", fontsize = 12)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                color="white" if cm[i,j] > 0.5 else "black")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(PLOT_DIR + "val_confusion_matrix.png", dpi=150, bbox_inches="tight")
print("Saved val_confusion_matrix.png")

## Plot: discriminant overlay — Nominal vs Hard-flip vs Pred-flip

In [ ]:
all_disc_cat = np.concatenate([disc, disc_flip, disc_pred_flip])
finite_all   = np.isfinite(all_disc_cat)
clip_disc    = np.percentile(np.abs(all_disc_cat[finite_all]), 100)
disc_bins    = np.linspace(-clip_disc + 5, clip_disc, 81)

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    fig = plt.figure(figsize=(7, 5))
    gs = fig.add_gridspec(2, 1, hspace=0.08, height_ratios=[3, 1])
    ax_main  = fig.add_subplot(gs[0])
    ax_ratio = fig.add_subplot(gs[1], sharex=ax_main)

    counts = {}
    for label, probs_c, d in CASES:
        finite = np.isfinite(d)
        m      = (all_true == cls_idx) & finite
        h, _   = np.histogram(d[m], bins=disc_bins, density=True)
        counts[label] = h
        ax_main.stairs(h, disc_bins, color=CASE_COLOURS[label],
                       linestyle=LINESTYLES[label], linewidth=3, label=label)

    ax_main.set_ylabel("Density", loc="top", fontsize=12)
    ax_main.set_yscale("log")
    ax_main.legend(fontsize=11)
    plt.setp(ax_main.get_xticklabels(), visible=False)

    h_nom = counts["Nominal"]
    valid_bin = h_nom > 0
    for label in ["Hard-flip", "Pred-flip"]:
        h_other = counts[label]
        ratio   = np.where(valid_bin, h_other / np.where(valid_bin, h_nom, 1), np.nan)
        ax_ratio.stairs(ratio, disc_bins, color=CASE_COLOURS[label],
                        linestyle=LINESTYLES[label], linewidth=2, label=label)
    ax_ratio.axhline(1.0, color="black", linewidth=0.8, linestyle="-")
    ax_ratio.set_ylim(0.5, 2.0)
    ax_ratio.set_ylabel("Flipped/ Nominal", fontsize=8)
    ax_ratio.yaxis.set_major_locator(plt.MultipleLocator(0.5))
    ax_ratio.tick_params(labelsize=7)
    ax_ratio.legend(fontsize=7)
    ax_ratio.set_xlabel(r"$D_{b}$", loc="right", fontsize=12)

    plt.tight_layout()
    fname = f"val_discriminant_overlay_{cls_name.replace('-', '_')}.png"
    plt.savefig(PLOT_DIR + fname, dpi=150, bbox_inches="tight")
    print(f"Saved {fname}")

## Plot: ROC overlay — Nominal vs Hard-flip vs Pred-flip

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
BKG_LINESTYLES = {"c-jet": "-", "light-jet": "--"}
BKG_COLOURS_ROC = {"c-jet": COLOURS["c-jet"], "light-jet": COLOURS["light-jet"]}
for bkg_idx, bkg_name in [(1, "c-jet"), (2, "light-jet")]:
    for label, probs_c, d in CASES:
        m      = (all_true == 0) | (all_true == bkg_idx)
        scores = (all_true[m] == 0).astype(int)
        score  = d[m]
        finite = np.isfinite(score)
        fpr, tpr, _ = roc_curve(scores[finite], score[finite])
        ax.plot(tpr, fpr, color=CASE_COLOURS[label], linewidth=3,
                label=f"b vs {bkg_name}, {label}  AUC={auc(fpr, tpr):.3f}",
                linestyle=BKG_LINESTYLES[bkg_name])
ax.set_xlabel("b-jet efficiency", loc="right", fontsize=12)
ax.set_ylabel("Background rate", loc="top", fontsize=12)
ax.set_yscale("log")
ax.legend(fontsize=9)
ax.grid(True, which="both", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(PLOT_DIR + "val_roc_overlay.png", dpi=150, bbox_inches="tight")
print("Saved val_roc_overlay.png")

## Plot: 2D scatter — Hard-flip vs Pred-flip discriminant (light jets)

In [ ]:
light_mask  = (all_true == 2) & np.isfinite(disc_flip) & np.isfinite(disc_pred_flip)
x_scatter   = disc_flip[light_mask]       # hard-flip
y_scatter   = disc_pred_flip[light_mask]  # pred-flip

clip_x = np.percentile(np.abs(x_scatter), 99)
clip_y = np.percentile(np.abs(y_scatter), 99)
lim    = max(clip_x, clip_y)

MAX_PTS = 50_000
if len(x_scatter) > MAX_PTS:
    rng_sc      = np.random.default_rng(0)
    sel         = rng_sc.choice(len(x_scatter), MAX_PTS, replace=False)
    x_sc, y_sc = x_scatter[sel], y_scatter[sel]
else:
    x_sc, y_sc = x_scatter, y_scatter

from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x_sc, y_sc, s=2, alpha=0.15, color=COLOURS["light-jet"], linewidths=0)

_kde      = gaussian_kde(np.vstack([x_sc, y_sc]))
_grid_pts = 200
_gx       = np.linspace(-lim, lim, _grid_pts)
_gy       = np.linspace(-lim, lim, _grid_pts)
_XX, _YY  = np.meshgrid(_gx, _gy)
_Z        = _kde(np.vstack([_XX.ravel(), _YY.ravel()])).reshape(_grid_pts, _grid_pts)
_z_flat   = np.sort(_Z.ravel())[::-1]
_cumsum   = np.cumsum(_z_flat) / _z_flat.sum()
_threshold = _z_flat[np.searchsorted(_cumsum, 0.90)]
ax.contour(_XX, _YY, _Z, levels=[_threshold], colors=["#d62728"], linewidths=1.5, linestyles="-")
ax.plot([], [], color="#d62728", linewidth=1.5, label="90% contour")

ax.axline((0, 0), slope=1, color="black", linewidth=0.8, linestyle="--", label="y = x")
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel(r"Hard-flip discriminant $D_{b}$", loc="right", fontsize=12)
ax.set_ylabel(r"Pred-flip discriminant $D_{b}$", loc="top", fontsize=12)
ax.legend(fontsize=12)
ax.set_aspect("equal")
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR + "val_disc_scatter_light_hardflip_vs_predflip.png", dpi=150, bbox_inches="tight")
print("Saved val_disc_scatter_light_hardflip_vs_predflip.png")

## Plot: tail zoom — light jet discriminant (Nominal vs Hard-flip vs Pred-flip)

In [ ]:
light_finite = (all_true == 2) & np.isfinite(disc) & np.isfinite(disc_flip) & np.isfinite(disc_pred_flip)
light_nom       = disc[light_finite]
light_hardflip  = disc_flip[light_finite]
light_predflip  = disc_pred_flip[light_finite]

for pct, pct_label, fname, n_bins in [
    (95,   "top 5%",   "val_disc_tail_zoom_light.png",      60),
    (99,   "top 1%",   "val_disc_tail_zoom_light_1pct.png", 20),
    (99.9, "top 0.1%", "val_disc_tail_zoom_light_01pct.png", 10),
]:
    tail_lo   = np.percentile(light_nom, pct)
    tail_hi   = max(np.percentile(light_nom, 99.99),
                    np.percentile(light_hardflip, 99.99),
                    np.percentile(light_predflip, 99.99))
    tail_bins = np.linspace(tail_lo, tail_hi, n_bins)

    fig = plt.figure(figsize=(7, 6))
    gs = fig.add_gridspec(2, 1, hspace=0.08, height_ratios=[3, 1])
    ax_main  = fig.add_subplot(gs[0])
    ax_ratio = fig.add_subplot(gs[1], sharex=ax_main)

    counts_t = {}
    for d_arr, label in [
        (light_nom,      "Nominal"),
        (light_hardflip, "Hard-flip"),
        (light_predflip, "Pred-flip"),
    ]:
        h, _ = np.histogram(d_arr, bins=tail_bins)
        counts_t[label] = h
        ax_main.stairs(h, tail_bins, linestyle=LINESTYLES[label], linewidth=3,
                       color=CASE_COLOURS[label], label=label)

    ax_main.set_ylabel("Counts", loc="top", fontsize=12)
    ax_main.legend(fontsize=11)
    ax_main.grid(True, linestyle="--", alpha=0.3)
    plt.setp(ax_main.get_xticklabels(), visible=False)

    h_nom_t = counts_t["Nominal"]
    valid_t = h_nom_t > 0
    for label in ["Hard-flip", "Pred-flip"]:
        ratio_t = np.where(valid_t, counts_t[label] / np.where(valid_t, h_nom_t, 1), np.nan)
        ax_ratio.stairs(ratio_t, tail_bins, color=CASE_COLOURS[label],
                        linestyle=LINESTYLES[label], linewidth=2, label=label)
    ax_ratio.axhline(1.0, color="black", linewidth=0.8, linestyle="-")
    ax_ratio.set_ylim(0.5, 2.0)
    ax_ratio.set_ylabel("Flipped/ Nominal", fontsize=8)
    ax_ratio.yaxis.set_major_locator(plt.MultipleLocator(0.5))
    ax_ratio.tick_params(labelsize=8)
    ax_ratio.grid(True, linestyle="--", alpha=0.3)
    ax_ratio.legend(fontsize=7)
    ax_ratio.set_xlabel(r"$D_{b}$", loc="right", fontsize=12)

    plt.tight_layout()
    plt.savefig(PLOT_DIR + fname, dpi=150, bbox_inches="tight")
    print(f"Saved {fname}")

## Table: mis-ID rates at fixed b-jet efficiency

Thresholds set by the **nominal** b-jet discriminant.

In [ ]:
B_EFFS = [0.90, 0.70, 0.50]

_nom_b_disc = disc[(all_true == 0) & np.isfinite(disc)]
THRESHOLDS  = {eff: np.percentile(_nom_b_disc, 100.0 * (1.0 - eff)) for eff in B_EFFS}

def binom_err(p, n):
    return np.sqrt(p * (1 - p) / n)

def stats_at_thresholds(disc_arr, true_arr, thresholds):
    b_disc     = disc_arr[(true_arr == 0) & np.isfinite(disc_arr)]
    light_disc = disc_arr[(true_arr == 2) & np.isfinite(disc_arr)]
    charm_disc = disc_arr[(true_arr == 1) & np.isfinite(disc_arr)]
    out = []
    for thr in thresholds.values():
        b_eff = (b_disc     >= thr).mean();  b_err = binom_err(b_eff, len(b_disc))
        l_eff = (light_disc >= thr).mean();  l_err = binom_err(l_eff, len(light_disc))
        c_eff = (charm_disc >= thr).mean();  c_err = binom_err(c_eff, len(charm_disc))
        out.append((b_eff, b_err, l_eff, l_err, c_eff, c_err))
    return out

nom_stats       = stats_at_thresholds(disc,           all_true, THRESHOLDS)
hard_flip_stats = stats_at_thresholds(disc_flip,      all_true, THRESHOLDS)
pred_flip_stats = stats_at_thresholds(disc_pred_flip, all_true, THRESHOLDS)

def fmt(v, e): return f"{v:.4%} ± {e:.4%}"
def fmt_cell(v, e): return f"{v:.2%} ± {e:.2%}"

col_labels = [
    "Nom b-eff",    "Hard b-eff",    "Pred b-eff",
    "Nom light",    "Hard light",    "Pred light",
    "Nom charm",    "Hard charm",    "Pred charm",
]

rows = []
for ns, hs, ps in zip(nom_stats, hard_flip_stats, pred_flip_stats):
    nb, nbe, nl, nle, nc, nce = ns
    hb, hbe, hl, hle, hc, hce = hs
    pb, pbe, pl, ple, pc2, pce = ps
    rows.append((nb,nbe, hb,hbe, pb,pbe,
                 nl,nle, hl,hle, pl,ple,
                 nc,nce, hc,hce, pc2,pce))

# terminal print
w = 24
header = f"{'WP':>6}  " + "  ".join(f"{c:>{w}}" for c in col_labels)
sep = "─" * len(header)
print("\n" + sep)
print(header)
print(sep)
for eff, r in zip(B_EFFS, rows):
    nb,nbe,hb,hbe,pb,pbe,nl,nle,hl,hle,pl,ple,nc,nce,hc,hce,pc2,pce = r
    vals = [fmt(nb,nbe),fmt(hb,hbe),fmt(pb,pbe),
            fmt(nl,nle),fmt(hl,hle),fmt(pl,ple),
            fmt(nc,nce),fmt(hc,hce),fmt(pc2,pce)]
    print(f"{eff:.0%}  " + "  ".join(f"{v:>{w}}" for v in vals))
print(sep)

# matplotlib table
fig, ax = plt.subplots(figsize=(22, 0.75 * (len(rows) + 2)))
ax.axis("off")
cell_text = []
for r in rows:
    nb,nbe,hb,hbe,pb,pbe,nl,nle,hl,hle,pl,ple,nc,nce,hc,hce,pc2,pce = r
    cell_text.append([
        fmt_cell(nb,nbe), fmt_cell(hb,hbe), fmt_cell(pb,pbe),
        fmt_cell(nl,nle), fmt_cell(hl,hle), fmt_cell(pl,ple),
        fmt_cell(nc,nce), fmt_cell(hc,hce), fmt_cell(pc2,pce),
    ])
row_labels = [f"{eff:.0%} WP" for eff in B_EFFS]
row_cols   = [["#f0f0f0" if i % 2 == 0 else "white"] * len(col_labels)
              for i in range(len(rows))]
tbl = ax.table(cellText=cell_text, colLabels=col_labels, rowLabels=row_labels,
               cellColours=row_cols, loc="center", cellLoc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.8)
fig.suptitle(
    f"Mis-identification rates at fixed b-jet efficiency\n"
    f"(thresholds set by nominal discriminant; pred-flip threshold={FLIP_THRESHOLD})",
    fontweight="bold", y=0.98,
)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "val_misid_table.png"), dpi=150, bbox_inches="tight")
print("Saved val_misid_table.png")
print(f"\nAll plots saved to {PLOT_DIR}")